# Experiment 3.0.3 — L3 bottleneck ablation

Analysis-only notebook. Training, checkpoint selection, frozen probes, and firing-rate extraction are executed by the Slurm pipeline. This notebook only reads finalized CSV artifacts, summarizes the results, and plots comparisons.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root(start=None):
    start = (Path.cwd() if start is None else Path(start)).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'snn').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate writingRing repository root')

REPO_ROOT = find_repo_root()
ARTIFACT_ROOT = REPO_ROOT / 'notebooks' / 'artifacts' / 'experiment_3_0_3_l3_bottleneck_ablation' / 'l3_bottleneck_v1'
print('Repository root:', REPO_ROOT)
print('Artifact root:', ARTIFACT_ROOT)


In [ ]:
files = {
    'architectures': 'experiment_3_0_3_architectures.csv',
    'native': 'experiment_3_0_3_native_results.csv',
    'native_summary': 'experiment_3_0_3_native_summary.csv',
    'history': 'experiment_3_0_3_history.csv',
    'layer_probes': 'experiment_3_0_3_layer_probes.csv',
    'layer_probe_summary': 'experiment_3_0_3_layer_probe_summary.csv',
    'subgroup_probes': 'experiment_3_0_3_tau_subgroup_probes.csv',
    'subgroup_probe_summary': 'experiment_3_0_3_tau_subgroup_probe_summary.csv',
    'firing': 'experiment_3_0_3_firing_rates.csv',
    'firing_summary': 'experiment_3_0_3_firing_rate_summary.csv',
    'diagnostics': 'experiment_3_0_3_diagnostics.csv',
    'diagnostic_summary': 'experiment_3_0_3_diagnostic_summary.csv',
}
missing = [name for name in files.values() if not (ARTIFACT_ROOT / name).exists()]
if missing:
    raise FileNotFoundError('Run/finalize Experiment 3.0.3 first. Missing: ' + ', '.join(missing))
frames = {key: pd.read_csv(ARTIFACT_ROOT / filename) for key, filename in files.items()}
architectures = frames['architectures']
native = frames['native']
native_summary = frames['native_summary']
history = frames['history']
layer_probes = frames['layer_probes']
layer_probe_summary = frames['layer_probe_summary']
subgroup_probes = frames['subgroup_probes']
subgroup_probe_summary = frames['subgroup_probe_summary']
firing = frames['firing']
firing_summary = frames['firing_summary']
diagnostics = frames['diagnostics']
diagnostic_summary = frames['diagnostic_summary']


## Architecture table


In [ ]:
display(architectures[['architecture','label','description','last_layer','layer','layer_width','shifts']].drop_duplicates())


## Native test balanced accuracy


In [ ]:
native_table = native_summary.pivot(index='architecture', columns='objective', values='mean_test_balanced_accuracy')
display(native_table)
ax = native_table.plot(kind='bar', figsize=(10,5))
ax.set_ylabel('Mean test balanced accuracy')
ax.set_title('Experiment 3.0.3 — Native Test BA')
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()


## Layer-wise fixed250 frozen linear probes

Use these plots to see whether L3 preserves, improves, or degrades the strong L2 representation. Architecture B has no L3.


In [ ]:
fixed = layer_probe_summary[layer_probe_summary['probe_type'] == 'fixed250'].copy()
for objective in fixed['objective'].unique():
    part = fixed[fixed['objective'] == objective]
    table = part.pivot(index='architecture', columns='layer', values='mean_probe_test_balanced_accuracy')
    display(table)
    ax = table.plot(kind='bar', figsize=(10,5))
    ax.set_title(f'{objective}: layer fixed250 probe Test BA')
    ax.set_ylabel('Balanced accuracy')
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()


## L3 preservation and native-readout utilization diagnostics

`last_minus_l2_fixed250_probe_ba` is L3 fixed250-probe BA minus L2 fixed250-probe BA for architectures with L3; negative means the third layer degraded linearly decodable temporal information. It is defined as zero for No-L3.

`readout_utilization_gap` is last-layer fixed250-probe BA minus native test BA.


In [ ]:
cols = [
    'objective','architecture','last_layer',
    'mean_l2_fixed250_probe_ba','mean_last_fixed250_probe_ba',
    'mean_last_minus_l2_fixed250_probe_ba','mean_test_balanced_accuracy',
    'mean_readout_utilization_gap'
]
display(diagnostic_summary[cols].sort_values(['objective','architecture']))
for metric, title in [
    ('mean_last_minus_l2_fixed250_probe_ba', 'Last layer minus L2 fixed250 probe BA'),
    ('mean_readout_utilization_gap', 'Frozen representation minus native BA'),
]:
    table = diagnostic_summary.pivot(index='architecture', columns='objective', values=metric)
    ax = table.plot(kind='bar', figsize=(10,5))
    ax.axhline(0, linewidth=1)
    ax.set_title(title)
    ax.set_ylabel('Balanced-accuracy difference')
    plt.tight_layout()
    plt.show()


## Tau-subgroup probes for multi-tau layers


In [ ]:
sub_fixed = subgroup_probe_summary[subgroup_probe_summary['probe_type'] == 'fixed250'].copy()
display(sub_fixed.sort_values(['objective','architecture','layer','shift']))


## Whole-layer firing rates on test split


In [ ]:
fr = firing_summary[(firing_summary['split'] == 'test') & (firing_summary['scope'] == 'whole_layer')].copy()
for objective in fr['objective'].unique():
    table = fr[fr['objective'] == objective].pivot(index='architecture', columns='layer', values='mean_firing_rate')
    display(table)
    ax = table.plot(kind='bar', figsize=(10,5))
    ax.set_title(f'{objective}: whole-layer test firing rate')
    ax.set_ylabel('Spikes / neuron / valid timestep')
    plt.tight_layout()
    plt.show()


## Validation BA trajectories


In [ ]:
mean_history = history.groupby(['objective','architecture','epoch'], as_index=False)['val_balanced_accuracy'].mean()
for objective in mean_history['objective'].unique():
    part = mean_history[mean_history['objective'] == objective]
    fig, ax = plt.subplots(figsize=(9,5))
    for architecture, group in part.groupby('architecture'):
        ax.plot(group['epoch'], group['val_balanced_accuracy'], label=architecture)
    ax.set_title(f'{objective}: mean validation BA vs epoch')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Validation balanced accuracy')
    ax.legend()
    plt.tight_layout()
    plt.show()


## Rankings


In [ ]:
display(native_summary.sort_values(['objective','mean_test_balanced_accuracy'], ascending=[True, False]))
display(diagnostic_summary.sort_values(['objective','mean_last_fixed250_probe_ba'], ascending=[True, False]))
